In [1]:
from datasets import load_dataset, load_from_disk
from replay.metrics import HitRate
import polars as pl
import pandas as pd
import json
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from torch.utils.tensorboard import SummaryWriter
import faiss
from functools import reduce
import datasets
import torch
import torch.nn as nn
from tqdm import tqdm
import os
from datetime import datetime

NUM_PROC = 32
CACHE_DIR = "/home/jupyter/filestore/storage/"

/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
2026-05-01 20:48:08.856998: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-01 20:48:11.958828: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
DATA_PATH = "/home/jupyter/filestore/storage/datasets/user_events_20230501"

dataset = load_from_disk(DATA_PATH)

In [3]:
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(-1)

polars.config.Config

In [4]:
polars_ds = dataset.to_polars()

In [5]:
TRAIN_END_DT = pd.to_datetime("2023-05-14")
TEST_END_DT = pd.to_datetime("2023-05-21")

train_interactions = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") <= TEST_END_DT)
    .with_columns(
        pl.col("item_condition_name").fill_null("NO_INFO"),
        pl.col("size_name").fill_null("NO_INFO"),
        pl.col("color").fill_null("NO_INFO")
    )
)

In [6]:
price_quantiles = []

price_stats = (
    train_interactions
    .select(
        pl.min("price").alias("min_price"),
        pl.max("price").alias("max_price")
    )
)

min_price = price_stats["min_price"][0]
max_price = price_stats["max_price"][0]

price_quantiles.append(min_price)
for q in np.arange(0.1, 1, 0.1):
    price_q = train_interactions.quantile(q, "nearest")["price"][0]
    price_quantiles.append(price_q)

price_quantiles.append(max_price)

print("quantiles: ", price_quantiles)
print("quantiles len: ", len(price_quantiles))

quantiles:  [1.0, 9.0, 12.0, 15.0, 20.0, 25.0, 34.0, 46.0, 70.0, 133.0, 5000.0]
quantiles len:  11


In [7]:
color2id = {value: idx for idx, value in enumerate(train_interactions.select("color").unique()["color"].to_list())}
condition2id = {value: idx for idx, value in enumerate(train_interactions.select("item_condition_name").unique()["item_condition_name"].to_list())}
size2id = {value: idx for idx, value in enumerate(train_interactions.select("size_name").unique()["size_name"].to_list())}
len(color2id.keys()), len(condition2id.keys()), len(size2id.keys())

(3929, 6, 333)

In [8]:
with open("data/color2id", "wb") as fp:
    pickle.dump(color2id, fp)
    
with open("data/condition2id", "wb") as fp:
    pickle.dump(condition2id, fp)
    
with open("data/size2id", "wb") as fp:
    pickle.dump(size2id, fp)

In [9]:
items = (
    train_interactions
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") <= TEST_END_DT)
    .filter(pl.col("event_id") == "item_like")
    .select("item_id", "name", "price", "item_condition_name", "size_name", "color")
)

all_negative_items = (
    items
    .unique()
    .with_columns(
        pl.col("item_condition_name").apply(lambda x: condition2id[x]),
        pl.col("size_name").apply(lambda x: size2id[x]),
        pl.col("color").apply(lambda x: color2id[x])
    )
    .join(
        items
        .groupby("item_id")
        .agg(pl.count("name").alias("likes_cnt")),
        on="item_id",
        how="left"
    )
)

item_interactions_counts = np.array(all_negative_items["likes_cnt"].to_list())

negatives_attrs = {
    "name": np.array(all_negative_items["name"].to_list()),
    "price": torch.tensor(all_negative_items["price"].to_list()),
    "item_condition_name": torch.tensor(all_negative_items["item_condition_name"].to_list()),
    "size_name": torch.tensor(all_negative_items["size_name"].to_list()),
    "color": torch.tensor(all_negative_items["color"].to_list())
}

In [10]:
sessions_with_two_likes = (
    train_interactions
    .filter(pl.col("event_id") == "item_like")
    .groupby("session_id", "user_id")
    .agg(pl.n_unique("item_id").alias("likes_cnt"))
    .filter(pl.col("likes_cnt") == 2)
)

sessions_with_two_likes.shape

(117721, 3)

In [11]:
sessions_to_take = (
    train_interactions
    .filter(pl.col("event_id") == "item_like")
    .join(
        sessions_with_two_likes,
        on=["session_id", "user_id"],
        how="inner"
    )
    .select(
        "session_id",
        "user_id",
        "item_id",
        "name",
        "price",
        "item_condition_name",
        "size_name",
        "color"
    )
)

In [12]:
aggregate_sessions = (
    sessions_to_take
    .with_columns(
        pl.col("item_condition_name").apply(lambda x: condition2id[x]),
        pl.col("size_name").apply(lambda x: size2id[x]),
        pl.col("color").apply(lambda x: color2id[x])
    )
    .unique()
    .groupby("user_id", "session_id")
    .agg(pl.struct("item_id", "name", "price", "item_condition_name", "size_name", "color").alias("item_attrs"))
)

In [13]:
from torch.utils.data import Dataset, DataLoader

In [14]:
class TrainDataset(Dataset):
    def __init__(self, sessions):
        self.sessions = sessions
        
    def __len__(self):
        return self.sessions.shape[0]
    
    def __getitem__(self, idx):
        left_item = self.sessions["item_attrs"][idx][0]
        right_item = self.sessions["item_attrs"][idx][1]
        return left_item, right_item

In [15]:
class ItemEmbedder(nn.Module):
    def __init__(self, text_encoder, price_quantiles, condition_num_values, size_num_values, color_num_values, emb_dim=64, device="cpu"):
        super().__init__()
        self.text_encoder = text_encoder
        self.device = device
        self.bins = torch.tensor(price_quantiles).to(device)
        self.bin_emb = nn.Embedding(len(price_quantiles), emb_dim)
        self.condition_emb = nn.Embedding(condition_num_values, 32)
        self.size_emb = nn.Embedding(size_num_values, 64)
        self.color_emb = nn.Embedding(color_num_values, 128)
        self.lin1 = nn.Linear(384 + 64 + 32 + 64 + 128, 256)
        self.lin2 = nn.Linear(256, 128)
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()
        
        
    def price_encoder(self, prices):
        B = prices.shape[0]
        idx = torch.bucketize(prices, self.bins, right=True)
        idx = torch.where(idx < len(self.bins), idx, idx - 1)
        left_border = self.bins[idx - 1]
        right_border = self.bins[idx]
        bins_len = right_border - left_border
        left_weights = (1 - (prices - left_border) / bins_len).reshape(B, 1)
        right_weights = (1 - (right_border - prices) / bins_len).reshape(B, 1)
        left_emb = left_weights * self.bin_emb(idx - 1)
        right_emb = right_weights * self.bin_emb(idx)
        price_emb = left_emb + right_emb
        return price_emb
    
    def forward(self, x):
        name_emb = self.text_encoder.encode(x["name"], batch_size=1024, normalize_embeddings=True, convert_to_tensor=True)
        price_emb = self.price_encoder(x["price"])
        condition_emb = self.condition_emb(x["item_condition_name"])
        size_emb = self.size_emb(x["size_name"])
        color_emb = self.color_emb(x["color"])
        item_emb = torch.cat([name_emb, price_emb, condition_emb, size_emb, color_emb], dim=-1).to(torch.float32)
        out_emb = self.lin2(self.relu(self.lin1(item_emb)))
        return out_emb

In [25]:
aggregate_sessions.shape

(117721, 3)

In [16]:
train_dataset = TrainDataset(aggregate_sessions.head(100000))
eval_dataset = TrainDataset(aggregate_sessions.tail(17721))

In [17]:
train_loader = DataLoader(train_dataset, batch_size=512)
eval_loader = DataLoader(eval_dataset, batch_size=512)

In [18]:
text_encoder = SentenceTransformer("all-MiniLM-L6-v2")

for p in text_encoder.parameters():
    p.requires_grad = False

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 634.67it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
model = ItemEmbedder(text_encoder, price_quantiles, len(condition2id.keys()), len(size2id.keys()), len(color2id.keys()), device="cuda:0")
print("model parameters count: ", sum([p.numel() for p in model.parameters() if p.requires_grad]))

model parameters count:  730304


In [20]:
def sample_negatives(positive_ids, item_interactions_counts, num_negatives=4000):
    probs = np.sqrt(item_interactions_counts)
    probs = probs / probs.sum()
    batch_size = positive_ids.shape[0]

    negative_samples = np.random.choice(
        item_interactions_counts.shape[0],
        size=(batch_size, num_negatives),
        p=probs
    )

    mask = (
        (negative_samples == positive_ids[:, 0:1]) |
        (negative_samples == positive_ids[:, 1:2])
    ).numpy()

    while mask.any():
        negative_samples[mask] = np.random.randint(
            0,
            all_negative_items.shape[0],
            size=mask.sum()
        )

        mask = (
            (negative_samples == positive_ids[:, 0:1]) |
            (negative_samples == positive_ids[:, 1:2])
        ).numpy()
    
    return negative_samples

In [21]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
writer = SummaryWriter()

In [22]:
def train_epoch(model, dataloader, writer, criterion, optimizer, epoch, num_negatives=100, top_k_negatives=10, device="cpu"):
    model.train()
    running_loss = 0
    for batch in tqdm(dataloader, total=len(dataloader), desc=f'Epoch: {epoch}, Training Batch', dynamic_ncols=True):
        left_items, right_items = batch
        B = left_items["item_id"].shape[0]
        positive_ids = torch.cat([left_items["item_id"].reshape(B, 1), right_items["item_id"].reshape(B, 1)], dim=-1)
        negative_ids = sample_negatives(positive_ids, item_interactions_counts, num_negatives=num_negatives)
        negative_items = {k: negatives_attrs[k][negative_ids].reshape(B * num_negatives) for k, _ in negatives_attrs.items()}
        for k in left_items.keys():
            if k not in ["name", "item_id"]:
                left_items[k] = left_items[k].to(device)
                right_items[k] = right_items[k].to(device)
                negative_items[k] = negative_items[k].to(device)
        left_embs = model(left_items)
        right_embs = model(right_items)
        neg_embs = model(negative_items)
        neg_embs = neg_embs.reshape(B, num_negatives, neg_embs.shape[-1])
        pos_scores = (left_embs * right_embs).sum(dim=-1).reshape(B, 1)
        neg_scores = (left_embs[:, None, :] * neg_embs).sum(dim=-1)#[0][:, :top_k_negatives]
        logits = torch.cat([pos_scores, neg_scores], dim=-1)
        y = torch.zeros(B, dtype=torch.long).to(device)
        loss = criterion(logits, y)
        if epoch > 0:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        running_loss += loss.detach().cpu().item()
    writer.add_scalar('Loss/Train', running_loss / len(dataloader), epoch + 1)
    return running_loss / len(dataloader), optimizer

In [23]:
def eval_epoch(model, dataloader, writer, criterion, optimizer, epoch, num_negatives=100, top_k_negatives=10, device="cpu"):
    model.eval()
    running_loss = 0
    for batch in tqdm(dataloader, total=len(dataloader), desc=f'Epoch: {epoch}, Eval Batch', dynamic_ncols=True):
        left_items, right_items = batch
        B = left_items["item_id"].shape[0]
        positive_ids = torch.cat([left_items["item_id"].reshape(B, 1), right_items["item_id"].reshape(B, 1)], dim=-1)
        negative_ids = sample_negatives(positive_ids, item_interactions_counts, num_negatives=num_negatives)
        negative_items = {k: negatives_attrs[k][negative_ids].reshape(B * num_negatives) for k, _ in negatives_attrs.items()}
        for k in left_items.keys():
            if k not in ["name", "item_id"]:
                left_items[k] = left_items[k].to(device)
                right_items[k] = right_items[k].to(device)
                negative_items[k] = negative_items[k].to(device)
        left_embs = model(left_items)
        right_embs = model(right_items)
        neg_embs = model(negative_items)
        neg_embs = neg_embs.reshape(B, num_negatives, neg_embs.shape[-1])
        pos_scores = (left_embs * right_embs).sum(dim=-1).reshape(B, 1)
        neg_scores = (left_embs[:, None, :] * neg_embs).sum(dim=-1)#[0][:, :top_k_negatives]
        logits = torch.cat([pos_scores, neg_scores], dim=-1)
        y = torch.zeros(B, dtype=torch.long).to(device)
        loss = criterion(logits, y)
        running_loss += loss.detach().cpu().item()
    writer.add_scalar('Loss/Eval', running_loss / len(dataloader), epoch + 1)
    return running_loss / len(dataloader)

In [24]:
def train(model, writer, train_dataloader, eval_dataloader, criterion, optimizer, num_negatives=10, top_k_negatives=10, epochs=20, device="cuda:0"):
    model = model.to(device)
    for i in range(epochs):
        train_loss, optimizer = train_epoch(
            model, train_dataloader, writer, criterion, optimizer, i,
            num_negatives=num_negatives, top_k_negatives=top_k_negatives, device=device
        )
        print("train loss: ", train_loss)
        eval_loss = eval_epoch(
            model, eval_dataloader, writer, criterion, optimizer, i,
            num_negatives=num_negatives, top_k_negatives=top_k_negatives, device=device
        )
        print("eval loss: ", eval_loss)

In [25]:
train(model, writer, train_loader, eval_loader, criterion, optimizer, device="cuda:0")

Epoch: 0, Training Batch: 100%|██████████| 196/196 [01:18<00:00,  2.49it/s]


train loss:  2.14892338854926


Epoch: 0, Eval Batch: 100%|██████████| 35/35 [00:13<00:00,  2.55it/s]


eval loss:  2.1518032210213796


Epoch: 1, Training Batch: 100%|██████████| 196/196 [01:18<00:00,  2.51it/s]


...

eval loss:  0.7161449330193655


Epoch: 19, Training Batch: 100%|██████████| 196/196 [01:17<00:00,  2.52it/s]


train loss:  0.5275068298286322


Epoch: 19, Eval Batch: 100%|██████████| 35/35 [00:13<00:00,  2.54it/s]

eval loss:  0.6877026813370841


In [28]:
torch.save(model.state_dict(), "data/item2item_v2.pth")

In [27]:
model

ItemEmbedder(
  (text_encoder): SentenceTransformer(
    (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
    (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
    (2): Normalize()
  )
  (bin_emb): Embedding(11, 64)
  (condition_emb): Embedding(6, 32)
  (size_emb): Embedding(333, 64)
  (color_emb): Embedding(3929, 128)
  (lin1): Linear(in_features=672, out_features=256, bias=True)
  (lin2): Linear(in_features=256, out_features=128, bias=True)
  (relu): ReLU()
  (tanh): Tanh()
)